# Memory-efficient Visa to ITL1 runner
Runs each stage separately and releases large dataframes after reduction.


In [ ]:
from pathlib import Path
import sys,gc,yaml,pandas as pd
from IPython.display import display
ROOT=Path.cwd(); sys.path.insert(0,str(ROOT/"src"))
from visa_itl1.io import read_table,write_table
from visa_itl1.memory import enable_copy_on_write,report
from visa_itl1.nspl import prepare_nspl
from visa_itl1.addressbase import prepare_commercial_property_counts
from visa_itl1.weights import match_counts_to_nspl,build_all_geography_weights
from visa_itl1.visa import prepare_visa,create_level_controls
from visa_itl1.allocation import allocate_level_controls
enable_copy_on_write()


In [ ]:
NSPL_PATH=ROOT/"data/nspl.parquet"
ADDRESSBASE_PATH=ROOT/"data/addressbase.parquet"
VISA_PATH=ROOT/"data/visa.parquet"
CONFIG_PATH=ROOT/"config/example.yml"
OUTPUT_DIR=ROOT/"outputs_notebook"
cfg=yaml.safe_load(CONFIG_PATH.read_text())


## 1. NSPL


In [ ]:
raw=read_table(NSPL_PATH); n=cfg["nspl"]
nspl,nspl_invalid=prepare_nspl(raw,postcode_column=n["postcode_column"],region_code_column=n["region_code_column"],region_names=cfg["region_names"])
del raw; gc.collect(); display(report("NSPL prepared",nspl=nspl))


## 2. Reduce AddressBase immediately


In [ ]:
ab=read_table(ADDRESSBASE_PATH); display(report("AddressBase loaded",addressbase=ab)); a=cfg["addressbase"]
postcode_counts,class_summary,address_diag=prepare_commercial_property_counts(ab,postcode_column=a["postcode_column"],classification_column=a.get("classification_column"),commercial_classification=a.get("commercial_classification","C"),missing_classification_policy=a.get("missing_classification_policy","exclude"),property_id_column=a.get("property_id_column"),deduplicate_properties=a.get("deduplicate_properties",False),input_mode=a.get("input_mode","raw"),count_column=a.get("count_column","commercial_property_count"))
del ab; gc.collect(); display(address_diag); display(report("AddressBase reduced",postcode_counts=postcode_counts))


## 3. Build compact weights


In [ ]:
matched,unmatched_address=match_counts_to_nspl(postcode_counts,nspl); del postcode_counts,nspl; gc.collect()
levels=cfg["allocation"].get("levels",["area","district","sector"])
weights,weight_diag=build_all_geography_weights(matched,levels,cfg["allocation"].get("weight_tolerance",1e-10)); del matched; gc.collect(); display(weight_diag.head()); display(report("Weights",weights=weights))


## 4. Prepare long-form Visa geography


In [ ]:
visa=read_table(VISA_PATH); v=cfg["visa"]
prepared,spend_quality,status=prepare_visa(visa,postcode_level_column=v["postcode_level_column"],postcode_code_column=v["postcode_code_column"],postcode_levels=v["postcode_levels"],spend_columns=v["spend_columns"],dimensions=v.get("dimensions",[]),suppressed_tokens=v.get("suppressed_tokens",[]),unknown_level_policy=v.get("unknown_level_policy","retain"))
del visa; gc.collect(); display(spend_quality); display(status); display(report("Visa prepared",prepared=prepared))


## 5. Allocate one level at a time


In [ ]:
results=[]; audits=[]
for level in levels:
 controls=create_level_controls(prepared,level=level,level_label=v["postcode_levels"][level],spend_columns=v["spend_columns"],dimensions=v.get("dimensions",[]))
 result,unmatched,audit=allocate_level_controls(controls,weights,level=level,spend_columns=v["spend_columns"],dimensions=v.get("dimensions",[]),unmatched_policy=cfg["allocation"].get("unmatched_policy","retain"),reconciliation_tolerance=cfg["allocation"].get("reconciliation_tolerance",1e-8))
 result["source_geography_level"]=level; results.append(result); audits.append(audit)
 write_table(unmatched,OUTPUT_DIR/"diagnostics"/f"visa_unmatched_{level}.parquet")
 del controls,result,unmatched,audit; gc.collect()
final=pd.concat(results,ignore_index=True); reconciliation=pd.concat(audits,ignore_index=True); display(reconciliation); display(final.head())


## 6. Write outputs


In [ ]:
write_table(final,OUTPUT_DIR/"main/visa_itl1_by_geography_level.parquet")
write_table(weights,OUTPUT_DIR/"lookups/postcode_itl1_weights.parquet")
write_table(unmatched_address,OUTPUT_DIR/"diagnostics/addressbase_unmatched_nspl.parquet")
write_table(class_summary,OUTPUT_DIR/"diagnostics/addressbase_classification_summary.csv")
write_table(address_diag,OUTPUT_DIR/"diagnostics/addressbase_reduction_summary.csv")
write_table(weight_diag,OUTPUT_DIR/"diagnostics/weight_validation.csv")
write_table(status,OUTPUT_DIR/"diagnostics/visa_geography_status.csv")
write_table(spend_quality,OUTPUT_DIR/"diagnostics/visa_spend_quality.csv")
write_table(reconciliation,OUTPUT_DIR/"reconciliation/reconciliation.csv")
print(OUTPUT_DIR.resolve())
